In [ ]:
# --- Section 0: Setup Project Environment (Kaggle) ---
import os
import sys
import shutil
import subprocess

# ── Step 1: Install lightweight dependencies only ───────────
# IMPORTANT: Do NOT reinstall torch/torchvision on Kaggle unlessจำเป็นจริง ๆ
# เพราะอาจทำให้ CUDA build ไม่ตรงกับ GPU ของเครื่องรัน
pkgs = [
    "transformers>=4.30",
    "matplotlib>=3.7",
    "pillow>=10.0",
    "numpy>=1.24",
    "scikit-learn>=1.3",
]

print("📦 Installing extra dependencies...")
cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-input", *pkgs]
subprocess.run(cmd, check=True)
print("✅ Install complete\n")

# ── Step 2: Copy project from Kaggle input dataset ──────────ฃ
SOURCE_DIR = "/kaggle/input/datasets/priyadatechasaratool/mae-classification-code01"  # เปลี่ยนเป็น dataset folder จริงของคุณ
WORK_DIR = "/kaggle/working/project"

if not os.path.exists(SOURCE_DIR):
    raise FileNotFoundError(
        f"SOURCE_DIR not found: {SOURCE_DIR}. "
        "Please update SOURCE_DIR to your Kaggle dataset folder."
    )

if not os.path.exists(WORK_DIR):
    print(f"📂 Copying files from\n  {SOURCE_DIR}\n→ {WORK_DIR} ...")
    shutil.copytree(SOURCE_DIR, WORK_DIR)
    print("✅ Copy complete!\n")
else:
    print(f"✅ Already exists: {WORK_DIR}\n")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f"📍 CWD: {os.getcwd()}")
print("📦 sys.path updated")
print("ℹ️ HF Hub warning about unauthenticated requests is normal if HF_TOKEN is not set.\n")

# ── Step 3: Verify key files ───────────────────────────────
key_files = [
    "start_implementation.ipynb",
    "data/animals10.py",
    "models/unet.py",
    "training/mae_trainer.py",
    "training/classification.py",
    "training/evaluation.py",
    "training/unet.py",
    "utils/common.py",
    "pyproject.toml",
]

print("🔍 Verifying key files...")
all_ok = True
for f in key_files:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {f}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ All files OK — ready to run training!")
else:
    print("\n⚠️ Some files are missing. Check your dataset upload.")

In [ ]:
# --- Section 1: Import Libraries and Configure Environment ---
from dataclasses import dataclass
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from utils.common import set_seed


@dataclass
class Config:
    DATA_ROOT: str = "/kaggle/input/datasets/alessiocorrado99/animals10/raw-img"
    CKPT_DIR: str = "/kaggle/working/checkpoints"
    RESULT_DIR: str = "/kaggle/working/results"
    BATCH_SIZE: int = 32
    EPOCHS_MAE: int = 50
    EPOCHS_UNET: int = 50
    EPOCHS_CLS: int = 20
    LR: float = 1e-4
    MASK_RATIO: float = 0.75
    IMG_SIZE: int = 224
    SEED: int = 42
    NUM_WORKERS: int = 2
    CLS_TRAIN_MODE: str = "partial"  # "partial" or "end_to_end"
    UNFREEZE_LAST_BLOCKS: int = 1
    UNFREEZE_VIT_LAYERNORM: bool = True
    USE_WEIGHTED_SAMPLER: bool = True
    EARLY_STOPPING_PATIENCE: int = 5
    EARLY_STOPPING_MIN_DELTA: float = 1e-4
    WARMUP_RATIO_MAE: float = 0.1
    WARMUP_RATIO_UNET: float = 0.1
    WARMUP_RATIO_CLS: float = 0.1


cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CUDA sanity check ว่่า GPU ทำงานได้: บางครั้ง torch build ไม่ตรงกับ GPU runtime บน Kaggle
if device.type == "cuda":
    try:
        _ = torch.zeros(1, device=device) + 1
        torch.cuda.synchronize()
    except Exception as exc:
        print(f"CUDA unavailable at runtime ({exc}). Fallback to CPU.")
        device = torch.device("cpu")

set_seed(cfg.SEED)

Path(cfg.CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(cfg.RESULT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
print(f"Dataset root: {cfg.DATA_ROOT}")
print(f"Checkpoint dir: {cfg.CKPT_DIR}")
print(f"Result dir: {cfg.RESULT_DIR}")
print(f"Classifier train mode: {cfg.CLS_TRAIN_MODE}")
print(f"Use weighted sampler: {cfg.USE_WEIGHTED_SAMPLER}")
print(f"Early stopping patience: {cfg.EARLY_STOPPING_PATIENCE}")
print(
    f"Warmup ratios (MAE/UNET/CLS): "
    f"{cfg.WARMUP_RATIO_MAE}/{cfg.WARMUP_RATIO_UNET}/{cfg.WARMUP_RATIO_CLS}"
)

In [ ]:
# --- Section 2: Load and Inspect the Dataset ---
from collections import Counter
import random
from PIL import Image

from data.animals10 import (
    build_split,
    discover_class_directories,
    map_animals10_labels_to_english,
)

class_dirs = discover_class_directories(cfg.DATA_ROOT)
split = build_split(cfg.DATA_ROOT, val_fraction=0.2, seed=cfg.SEED)
idx_to_class = {index: class_name for class_name, index in split.class_to_idx.items()}

class_to_idx_en, idx_to_class_en = map_animals10_labels_to_english(split.class_to_idx)

print(f"Classes found: {len(class_dirs)}")
print("Original folder names (IT):", sorted(split.class_to_idx.keys()))
print("Classification labels (EN):", [idx_to_class_en[i] for i in sorted(idx_to_class_en.keys())])
print(f"Train samples: {len(split.train_samples)}")
print(f"Val samples: {len(split.val_samples)}")

train_counts = Counter(label for _, label in split.train_samples)
val_counts = Counter(label for _, label in split.val_samples)
print("Train distribution:", {idx_to_class_en[idx]: count for idx, count in train_counts.items()})
print("Val distribution:", {idx_to_class_en[idx]: count for idx, count in val_counts.items()})

# สุ่มตัวอย่าง 4 ภาพจาก train set มาโชว์
random_samples = random.sample(split.train_samples, 4)
sample_paths = [path for path, _ in random_samples]

fig, axes = plt.subplots(1, len(sample_paths), figsize=(16, 4))
for axis, sample_path in zip(axes, sample_paths):
    with Image.open(sample_path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(Path(sample_path).parent.name)
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- Section 3: Define Image Preprocessing and Augmentation ---
from torchvision import transforms as T
from data.animals10 import IMAGENET_MEAN, IMAGENET_STD

train_transform = T.Compose(
    [
        T.RandomResizedCrop(
            cfg.IMG_SIZE,
            scale=(0.2, 1.0),
            interpolation=T.InterpolationMode.BICUBIC,
        ),
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

val_transform = T.Compose(
    [
        T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(cfg.IMG_SIZE),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

print(train_transform)
print(val_transform)

In [ ]:
# --- Section 4: Build Data Loaders ---
from data.animals10 import build_dataloaders

train_loader, val_loader, split = build_dataloaders(
    root=cfg.DATA_ROOT,
    batch_size=cfg.BATCH_SIZE,
    val_fraction=0.2,
    seed=cfg.SEED,
    num_workers=cfg.NUM_WORKERS,
    train_transform=train_transform,
    val_transform=val_transform,
    use_weighted_sampler=cfg.USE_WEIGHTED_SAMPLER,
)

idx_to_class = {index: class_name for class_name, index in split.class_to_idx.items()}
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Train sampler: {type(train_loader.sampler).__name__}")
first_batch = next(iter(train_loader))
print(f"First batch image shape: {tuple(first_batch[0].shape)}")
print(f"First batch label shape: {tuple(first_batch[1].shape)}")

In [ ]:
# --- Section 5: Implement the MAE Encoder-Decoder Model ---
from training.mae_trainer import load_mae_model, load_mae_processor, reconstruct_image, reconstruct_mae_images

mae_processor = load_mae_processor()
mae_model = load_mae_model(mask_ratio=cfg.MASK_RATIO).to(device)
mae_model.eval()

sample_images = first_batch[0][:2].to(device) # ใช้แค่ 2 ภาพแรกจาก batch แรกเพื่อทดสอบโมเดล
with torch.no_grad():
    try:
        mae_outputs = mae_model(pixel_values=sample_images)
        mae_loss = mae_outputs.loss.item()
    except RuntimeError as exc:
        error_text = str(exc).lower()
        if "no kernel image is available" in error_text or "cuda error" in error_text:
            print("⚠️ CUDA kernel compatibility issue detected. Switching to CPU runtime.")
            device = torch.device("cpu")
            mae_model = mae_model.to(device)
            sample_images = sample_images.to(device)
            mae_outputs = mae_model(pixel_values=sample_images)
            mae_loss = mae_outputs.loss.item()
        else:
            raise

print(f"MAE loss on first batch: {mae_loss:.6f}")
print("MAE model ready for continual pre-training and reconstruction.")

In [ ]:
# --- Section 6: Add the Classification Head ---
from transformers import ViTForImageClassification
from training.classification import load_mae_encoder_weights_into_classifier, set_classifier_train_mode

label2id = class_to_idx_en
id2label = {idx: label for idx, label in idx_to_class_en.items()}

classification_model = ViTForImageClassification.from_pretrained(
    "facebook/vit-mae-base",
    num_labels=len(split.class_to_idx),
    ignore_mismatched_sizes=True,
    label2id=label2id,
    id2label=id2label,
).to(device)

classification_model = load_mae_encoder_weights_into_classifier(
    classification_model,
    mae_checkpoint_path=None,
)

trainable_params = set_classifier_train_mode(
    classification_model,
    mode=cfg.CLS_TRAIN_MODE,
    unfreeze_last_blocks=cfg.UNFREEZE_LAST_BLOCKS,
    unfreeze_vit_layernorm=cfg.UNFREEZE_VIT_LAYERNORM,
)

print("Classification head attached.")
print(classification_model.__class__.__name__)
print("Model labels (EN):", classification_model.config.id2label)
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# --- Section 7: Define Loss Functions and Optimizer ---
from torch.optim.lr_scheduler import ReduceLROnPlateau

from models.unet import UNet
from utils.common import create_grad_scaler
from training.unet import apply_patch_mask

unet_model = UNet().to(device)
mae_optimizer = torch.optim.AdamW(mae_model.parameters(), lr=cfg.LR)
unet_optimizer = torch.optim.AdamW(unet_model.parameters(), lr=cfg.LR)
# Optimize only parameters marked as trainable by CLS_TRAIN_MODE
cls_optimizer = torch.optim.AdamW((p for p in classification_model.parameters() if p.requires_grad), lr=cfg.LR)

mae_scheduler = ReduceLROnPlateau(
    mae_optimizer,
    mode="min",
    factor=cfg.LR_PLATEAU_FACTOR,
    patience=cfg.LR_PLATEAU_PATIENCE,
    min_lr=cfg.LR_PLATEAU_MIN_LR,
)
unet_scheduler = ReduceLROnPlateau(
    unet_optimizer,
    mode="min",
    factor=cfg.LR_PLATEAU_FACTOR,
    patience=cfg.LR_PLATEAU_PATIENCE,
    min_lr=cfg.LR_PLATEAU_MIN_LR,
)
cls_scheduler = ReduceLROnPlateau(
    cls_optimizer,
    mode="max",
    factor=cfg.LR_PLATEAU_FACTOR,
    patience=cfg.LR_PLATEAU_PATIENCE,
    min_lr=cfg.LR_PLATEAU_MIN_LR,
)

mae_scaler = create_grad_scaler(device)
unet_scaler = create_grad_scaler(device)
cls_scaler = create_grad_scaler(device)

print("Optimizers, schedulers, and mixed precision scaler are ready.")

In [ ]:
# --- Section 8: Train the Model (REAL TRAINING LOOP) ---
from training.mae_trainer import MAETrainer
from training.unet import UNetReconstructionTrainer
from training.classification import ViTClassifierTrainer
from utils.common import EarlyStopping

# 1. 🚀 เทรน MAE ให้รู้จักโครงสร้างภาพ
print(f"--- Starting MAE Pre-training ({cfg.EPOCHS_MAE} Epochs) ---")
mae_trainer = MAETrainer(mae_model, mae_optimizer, device, mask_ratio=cfg.MASK_RATIO, scaler=mae_scaler)
mae_early_stopper = EarlyStopping(
    patience=cfg.EARLY_STOPPING_PATIENCE,
    min_delta=cfg.EARLY_STOPPING_MIN_DELTA,
    mode="min",
)
best_mae_val = float("inf")

for epoch in range(cfg.EPOCHS_MAE):
    mae_train_loss = mae_trainer.train_epoch(train_loader)
    mae_val_loss = mae_trainer.evaluate_epoch(val_loader)
    mae_scheduler.step(mae_val_loss)

    mae_metrics = {
        "train_loss": mae_train_loss,
        "val_loss": mae_val_loss,
        "lr": mae_optimizer.param_groups[0]["lr"],
    }
    mae_trainer.save_checkpoint(
        Path(cfg.CKPT_DIR) / f"mae_epoch_{epoch+1:03d}.pt",
        epoch=epoch+1,
        metrics=mae_metrics,
        config={"mask_ratio": cfg.MASK_RATIO},
    )

    if mae_val_loss < best_mae_val:
        best_mae_val = mae_val_loss
        mae_trainer.save_checkpoint(
            Path(cfg.CKPT_DIR) / "mae_best.pt",
            epoch=epoch+1,
            metrics=mae_metrics,
            config={"mask_ratio": cfg.MASK_RATIO},
        )

    print(
        f"MAE Epoch {epoch+1}/{cfg.EPOCHS_MAE} | "
        f"Train Loss: {mae_train_loss:.4f} | Val Loss: {mae_val_loss:.4f} | "
        f"LR: {mae_optimizer.param_groups[0]['lr']:.2e}"
    )

    if mae_early_stopper.step(mae_val_loss):
        print(f"Early stopping MAE at epoch {epoch+1}")
        break

torch.save(mae_model.state_dict(), f"{cfg.CKPT_DIR}/mae_final.pt")

# 2. 🚀 เทรน U-Net ให้ซ่อมภาพแข่งกับ MAE
print(f"\n--- Starting U-Net Training ({cfg.EPOCHS_UNET} Epochs) ---")
unet_trainer = UNetReconstructionTrainer(
    unet_model,
    unet_optimizer,
    device,
    mask_ratio=cfg.MASK_RATIO,
    scaler=unet_scaler,
 )
unet_early_stopper = EarlyStopping(
    patience=cfg.EARLY_STOPPING_PATIENCE,
    min_delta=cfg.EARLY_STOPPING_MIN_DELTA,
    mode="min",
)
best_unet_val = float("inf")

for epoch in range(cfg.EPOCHS_UNET):
    unet_train_loss = unet_trainer.train_epoch(train_loader)
    unet_val_loss = unet_trainer.evaluate_epoch(val_loader)
    unet_scheduler.step(unet_val_loss)

    unet_metrics = {
        "train_loss": unet_train_loss,
        "val_loss": unet_val_loss,
        "lr": unet_optimizer.param_groups[0]["lr"],
    }
    unet_trainer.save_checkpoint(
        Path(cfg.CKPT_DIR) / f"unet_epoch_{epoch+1:03d}.pt",
        epoch=epoch+1,
        metrics=unet_metrics,
        config={"mask_ratio": cfg.MASK_RATIO},
    )

    if unet_val_loss < best_unet_val:
        best_unet_val = unet_val_loss
        unet_trainer.save_checkpoint(
            Path(cfg.CKPT_DIR) / "unet_best.pt",
            epoch=epoch+1,
            metrics=unet_metrics,
            config={"mask_ratio": cfg.MASK_RATIO},
        )

    print(
        f"U-Net Epoch {epoch+1}/{cfg.EPOCHS_UNET} | "
        f"Train Loss: {unet_train_loss:.4f} | Val Loss: {unet_val_loss:.4f} | "
        f"LR: {unet_optimizer.param_groups[0]['lr']:.2e}"
    )

    if unet_early_stopper.step(unet_val_loss):
        print(f"Early stopping U-Net at epoch {epoch+1}")
        break

torch.save(unet_model.state_dict(), f"{cfg.CKPT_DIR}/unet_final.pt")

# 3. 🚀 เทรน Classification ทายชื่อสัตว์
print(f"\n--- Starting Classification Training ({cfg.EPOCHS_CLS} Epochs) ---")
cls_trainer = ViTClassifierTrainer(classification_model, cls_optimizer, device, scaler=cls_scaler)
cls_early_stopper = EarlyStopping(
    patience=cfg.EARLY_STOPPING_PATIENCE,
    min_delta=cfg.EARLY_STOPPING_MIN_DELTA,
    mode="max",
)
best_cls_val_acc = float("-inf")

for epoch in range(cfg.EPOCHS_CLS):
    cls_train_loss, cls_train_acc = cls_trainer.train_epoch(train_loader)
    cls_val_loss, cls_val_acc = cls_trainer.evaluate_epoch(val_loader)
    cls_scheduler.step(cls_val_acc)

    cls_metrics = {
        "train_loss": cls_train_loss,
        "train_acc": cls_train_acc,
        "val_loss": cls_val_loss,
        "val_acc": cls_val_acc,
        "lr": cls_optimizer.param_groups[0]["lr"],
    }
    cls_trainer.save_checkpoint(
        Path(cfg.CKPT_DIR) / f"cls_epoch_{epoch+1:03d}.pt",
        epoch=epoch+1,
        metrics=cls_metrics,
        config={"train_mode": cfg.CLS_TRAIN_MODE},
    )

    if cls_val_acc > best_cls_val_acc:
        best_cls_val_acc = cls_val_acc
        cls_trainer.save_checkpoint(
            Path(cfg.CKPT_DIR) / "cls_best.pt",
            epoch=epoch+1,
            metrics=cls_metrics,
            config={"train_mode": cfg.CLS_TRAIN_MODE},
        )

    print(
        f"CLS Epoch {epoch+1}/{cfg.EPOCHS_CLS} | "
        f"Train Acc: {cls_train_acc:.4f} | Val Acc: {cls_val_acc:.4f} | "
        f"LR: {cls_optimizer.param_groups[0]['lr']:.2e}"
    )

    if cls_early_stopper.step(cls_val_acc):
        print(f"Early stopping Classifier at epoch {epoch+1}")
        break

torch.save(classification_model.state_dict(), f"{cfg.CKPT_DIR}/cls_final.pt")
print("\n🎉 Training Complete! All models saved.")

In [ ]:
# --- Section 9: Evaluate Reconstruction and Classification Metrics ---
from training.evaluation import compare_reconstruction_on_batch, evaluate_classifier

comparison_batch = next(iter(val_loader))
comparison_image_path = Path(cfg.RESULT_DIR) / "comparison" / "sample_comparison.png"
reconstruction_metrics = compare_reconstruction_on_batch(
    mae_model=mae_model,
    unet_model=unet_model,
    batch=comparison_batch,
    device=device,
    mask_ratio=cfg.MASK_RATIO,
    output_path=comparison_image_path,
)
classification_metrics = evaluate_classifier(classification_model, val_loader, device)

print("Reconstruction metrics:", reconstruction_metrics)
print("Classification metrics:", classification_metrics)

In [ ]:
# --- Section 10: Save Model Checkpoints and Inference Outputs ---
from utils.common import save_json

summary_payload = {
    "device": str(device),
    "data_root": cfg.DATA_ROOT,
    "class_labels_en": [idx_to_class_en[i] for i in sorted(idx_to_class_en.keys())],
    "mae_val_mse": float(mae_val_loss),
    "unet_val_mse": float(unet_val_loss),
    "classification_val_loss": float(cls_val_loss),
    "classification_val_accuracy": float(cls_val_acc),
    "comparison_image_path": str(comparison_image_path),
}

save_json(Path(cfg.RESULT_DIR) / "run_summary.json", summary_payload)
print("Saved summary to:", Path(cfg.RESULT_DIR) / "run_summary.json")
print("Comparison image saved to:", comparison_image_path)
print("Checkpoint directories:", cfg.CKPT_DIR)